# Focusing grating-coupler PCell: fiber to chip

This example follows [FDTD-Bench problem 07](https://github.com/doplaydo/FDTD-Bench/tree/main/problems/07_focusing_grating_coupler), based on the [Tidy3D FocusedApodGC example](https://www.flexcompute.com/tidy3d/examples/notebooks/FocusedApodGC/), using a parameterized GDSFactory cell instead of importing a GDS file. The cell uses generic-PDK mask layers, while the source placement and analytic incident-power normalization follow the benchmark setup.

In [ ]:
from pathlib import Path
from runpy import run_path

import gdsfactory as gf

from gsim import fdtd

PCELL_PATH = Path("gds/build_focusing_gc.py")
if not PCELL_PATH.is_file():
    PCELL_PATH = Path("nbs") / PCELL_PATH

STRAIGHT_WAVEGUIDE_LENGTH_UM = 2.0
FIBER_CENTER_UM = (22.4041, 0.0, 1.05)
FIBER_SIZE_UM = (12.48, 12.48, 0.0)
FIBER_DIRECTION = (-0.17388, 0.0, -0.98477)
DEVICE_CENTER_X_UM = 17.136
DEVICE_SPAN_X_UM = 34.272
DEVICE_SPAN_Y_UM = 23.546

## Build the component with the generic PDK

The PCell draws the full silicon footprint on `WG` and the grating gaps on `DEEP_ETCH`. The generic-PDK layer stack converts those masks into 220 nm silicon with a 130 nm slab in the etched regions.

In [ ]:
pdk = gf.gpdk.PDK
pdk.activate()
focusing_grating_coupler = run_path(PCELL_PATH)["focusing_grating_coupler"]
component = focusing_grating_coupler(waveguide_length_um=STRAIGHT_WAVEGUIDE_LENGTH_UM)
component

## Configure the in-coupling simulation

The source coordinates and vectors follow the FDTD-Bench configuration, translated with the shortened input waveguide. Gmsh preserves geometry to 20 nm, independently of the 50 nm Yee-grid cell size.

In [ ]:
simulation = fdtd.Simulation(pdk=pdk)
simulation.materials(background="sio2", overrides={"si": 3.47, "sio2": 1.44})
simulation.geometry(component, mesh_size_nm=500, geometry_tolerance_nm=20)
simulation.source = fdtd.GaussianBeamSource(
    center_um=FIBER_CENTER_UM,
    size_um=FIBER_SIZE_UM,
    aperture_normal="-z",
    propagation_direction=FIBER_DIRECTION,
    e_polarization=(0.0, 1.0, 0.0),
    focal_point_um=FIBER_CENTER_UM,
    waist_radius_um=5.2,
    refractive_index=1.44,
    wavelength_um=1.55,
    wavelength_span_um=0.1,
    num_wavelengths=101,
)
simulation.monitors.add_plane(
    name="cross",
    center_um=(DEVICE_CENTER_X_UM, 0.0, 0.0),
    size_um=(DEVICE_SPAN_X_UM, 0.0, 4.0),
    normal="+y",
    flux=False,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.55]),
)
simulation.monitors.add_plane(
    name="top_down",
    center_um=(DEVICE_CENTER_X_UM, 0.0, 0.065),
    size_um=(DEVICE_SPAN_X_UM, DEVICE_SPAN_Y_UM, 0.0),
    normal="+z",
    flux=False,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.55]),
)
simulation.monitors.add_plane(
    name="src_down",
    center_um=(FIBER_CENTER_UM[0], 0.0, 0.4),
    size_um=FIBER_SIZE_UM,
    normal="-z",
    flux=True,
)
simulation.domain(padding_um=0.5)
simulation.solver(cell_size_nm=50, energy_decay_fraction=1e-5)

In [ ]:
simulation.plot_3d()

## Run and normalize

A Gaussian source has no incident eigenmode coefficient, so `port_outputs` is not an S-parameter. The helper divides the raw `o1` modal power by the analytic Gaussian pulse power used by ZapFDTD and masks samples more than 25 dB below the pulse peak.

In [ ]:
result = simulation.run(check_cache=True)

In [ ]:
coupling = fdtd.gaussian_coupling_efficiency(result, simulation.source, port="o1")
coupling.plot_plotly()

## Inspect the fields

In [ ]:
result.monitors["cross"].plot_heatmap(
    wavelength_um=1.55, quantity="abs_e", cmap="magma"
)

In [ ]:
result.monitors["top_down"].plot_heatmap(
    wavelength_um=1.55, quantity="abs_e", cmap="magma"
)

The `src_down` flux monitor remains available as a diagnostic, but its net flux is not used as the incident-power normalization because reflected fields also cross that plane.